# Полные предложения с подвохами на больших моделях

Прогон `scripts/sentences_tricks.py` (протокол записан в начале файла и закоммичен до прогона) на моделях разных семейств и размеров. Интерпретатор здесь не обучается: его веса лежат в репозитории (`results/sentences_tricks_interpreter.pt`), поэтому все модели читаются одними и теми же весами.

**Перед запуском:** «Среда выполнения» → «Сменить среду выполнения» → GPU (T4). Затем ячейки по очереди. Каждый прогон пишет свой файл и не перезаписывает существующий.

In [ ]:
!nvidia-smi

Код скачивается прямо из репозитория, вместе с весами интерпретатора.

In [ ]:
!rm -rf whybox && git clone -q https://github.com/knbww/whybox.git
%cd whybox

In [ ]:
!pip install -q "transformers>=4.56" pyyaml pytest bitsandbytes accelerate

Проверка, что предложения грамматичны до и после каждого подавления.

In [ ]:
!python -m pytest -q tests/test_tricks.py

## Другие семейства, 1–1,7 млрд параметров (float32)

Pythia-1.4B, SmolLM2-1.7B, OLMo-2-1B. Около 15 минут.

In [ ]:
!PYTHONPATH=.:src:scripts python scripts/sentences_tricks.py \
    --models EleutherAI/pythia-1.4b HuggingFaceTB/SmolLM2-1.7B allenai/OLMo-2-0425-1B \
    --device cuda --dtype float32 \
    --out results/sentences_tricks_colab_families.json

## 2,7–2,8 млрд параметров (float16)

Pythia-2.8B и Phi-2. Около 10 минут.

In [ ]:
!PYTHONPATH=.:src:scripts python scripts/sentences_tricks.py \
    --models EleutherAI/pythia-2.8b microsoft/phi-2 \
    --device cuda --dtype float16 \
    --out results/sentences_tricks_colab_fp16.json

## 7 млрд параметров (4 бита)

Pythia-6.9B и Qwen2.5-7B. С градиентами в T4 они помещаются только в 4-битном виде; хватает ли точности, покажет ошибка полноты в выводе. Около 20–25 минут.

In [ ]:
!PYTHONPATH=.:src:scripts python scripts/sentences_tricks.py \
    --models EleutherAI/pythia-6.9b Qwen/Qwen2.5-7B-Instruct \
    --device cuda --load-in-4bit \
    --out results/sentences_tricks_colab_7b.json

## Итог и скачивание результатов

In [ ]:
import glob, json
from google.colab import files
new = ['families', 'fp16', '7b']
for f in sorted(glob.glob('results/sentences_tricks_colab_*.json')):
    if not any(n in f for n in new):
        continue
    d = json.load(open(f))
    for model, by_t in d['results'].items():
        for t, r in by_t.items():
            acc = {k: round(v, 3) for k, v in r['arms']['interpreter'].items() if v == v}
            print(f"{model:30s} {t:8s} {r['read_out']:26s} interpreter {acc}  "
                  f"live {r['arms']['live']['ALL']:.3f}  completeness {r['completeness_residual'][0]:.4f}")
    files.download(f)